# 1. Detecção de Objetos com YOLOv8 (Ultralytics)

## 1.1 Objetivo
Este notebook tem como objetivo ensinar o processo de **detecção de objetos em tempo real** utilizando o modelo YOLOv8 da biblioteca Ultralytics. Você aprenderá a:
1. Carregar modelos pré-treinados.
2. Realizar inferência em imagens e vídeos.
3. Capturar vídeo da webcam e processá-lo em tempo real.
4. Extrair e utilizar informações detalhadas das detecções (classe, confiança, coordenadas).
5. Criar lógica personalizada baseada nas detecções.

## 1.2 O que é Detecção de Objetos?
A detecção de objetos é uma tarefa de visão computacional que envolve identificar e localizar objetos de certas classes dentro de uma imagem ou vídeo. Diferente da classificação de imagem simples, que diz *o que* está na imagem, a detecção diz *o que* e *onde* (através de caixas delimitadoras ou bounding boxes).

## 1.3 Ferramentas Utilizadas

### Ultralytics YOLO
A biblioteca `ultralytics` fornece uma interface fácil de usar para os modelos YOLO (You Only Look Once). O YOLO é famoso por sua velocidade e precisão, sendo capaz de processar imagens em tempo real.

### OpenCV (cv2)
O `opencv-python` é uma biblioteca fundamental para processamento de imagens. Neste notebook, a utilizaremos para:
- Ler imagens e vídeos.
- Acessar a webcam.
- Desenhar caixas e textos nas imagens.
- Exibir os resultados na tela.

# 2. Primeiros Passos com Imagens e Vídeos

## 2.1 Inferência em Imagens
Primeiro, vamos carregar o modelo e testá-lo em uma imagem estática. O modelo `yolo11n.pt` é uma versão 'nano' (leve e rápida) do YOLO11.

In [1]:
from ultralytics import YOLO

# Carregar um modelo YOLO11n pré-treinado
model = YOLO("yolo11n.pt")

# Definir caminho para o arquivo de imagem
source = r"amostras\image-do.jpg"

# Executar inferência na origem
results = model(source, save=True)


image 1/1 c:\Users\dario\Documents\GitHub\machine_learning\aulas\03_Vision\aula_49_18-12\YOLO\amostras\image-do.jpg: 384x640 15 persons, 1 backpack, 1 suitcase, 1 potted plant, 58.1ms
Speed: 1.7ms preprocess, 58.1ms inference, 10.1ms postprocess per image at shape (1, 3, 384, 640)
Results saved to C:\Users\dario\Documents\GitHub\machine_learning\runs\detect\predict6


## 2.2 Inferência em Vídeos
O processo para vídeos é muito semelhante. A principal diferença é que o modelo processará cada quadro (frame) do vídeo sequencialmente.

In [2]:
from ultralytics import YOLO

# Carregar um modelo YOLO11n pré-treinado
model = YOLO("yolo11n.pt")

# Definir caminho para o arquivo de vídeo
source = r"amostras\video-do.mp4"

# Executar inferência na origem
# O argumento 'conf' define a confiança mínima para considerar uma detecção
results = model(source, conf=0.40, save=True)


WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/342) c:\Users\dario\Documents\GitHub\machine_learning\aulas\03_Vision\aula_49_18-12\YOLO\amostras\video-do.mp4: 384x640 13 persons, 10.9ms
video 1/1 (frame 2/342) c:\Users\dario\Documents\GitHub\machine_learning\aulas\03_Vision\aula_49_18-12\YOLO\amostras\video-do.mp4: 384x640 13 persons, 7.3ms
video 1/1 (frame 3/342) c:\Users\dario\Documents\GitHub\machine_learning\aulas\03_Vision\aula_49_18-12\YOLO\amostras\video-do.mp4: 384x640 12 pers

# 3. Detecção em Tempo Real (Webcam)

Para trabalhar em tempo real, precisamos de um loop contínuo que captura frames da câmera, envia para o modelo e exibe o resultado.

## 3.1 Loop de Inferência
O código abaixo realiza os seguintes passos repetidamente:
1. `cap.read()`: Captura um quadro da webcam.
2. `model(frame)`: O YOLO processa esse quadro.
3. `results[0].plot()`: A biblioteca desenha as detecções na imagem.
4. `cv2.imshow()`: Mostra a imagem processada numa janela.

In [3]:
import cv2
from ultralytics import YOLO

# Carregar o modelo YOLOv11 pré-treinado
model = YOLO('yolo11n.pt')

# Inicializar a captura de vídeo da webcam (dispositivo padrão, índice 0)
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Erro ao acessar a webcam.")
    exit()

while True:
    # Capturar frame da webcam
    ret, frame = cap.read()
    if not ret:
        print("Falha ao capturar imagem.")
        break

    # Realizar inferência no frame capturado
    results = model(frame)

    # Desenhar as detecções no frame (bounding boxes e labels)
    annotated_frame = results[0].plot()

    # Exibir o frame anotado na tela
    cv2.imshow('Detecção em Tempo Real', annotated_frame)

    # Sair do loop ao pressionar a tecla 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Liberar a câmera e fechar janelas
cap.release()
cv2.destroyAllWindows()


0: 480x640 5 persons, 1 cup, 1 chair, 3 laptops, 44.1ms
Speed: 1.4ms preprocess, 44.1ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 5 persons, 3 chairs, 4 laptops, 17.5ms
Speed: 2.1ms preprocess, 17.5ms inference, 2.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 6 persons, 1 chair, 2 laptops, 17.8ms
Speed: 1.7ms preprocess, 17.8ms inference, 2.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 5 persons, 2 chairs, 3 laptops, 16.4ms
Speed: 1.9ms preprocess, 16.4ms inference, 2.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 5 persons, 2 chairs, 3 laptops, 16.2ms
Speed: 1.7ms preprocess, 16.2ms inference, 2.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 5 persons, 1 chair, 1 dining table, 3 laptops, 17.3ms
Speed: 1.7ms preprocess, 17.3ms inference, 2.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 5 persons, 2 chairs, 1 dining table, 2 laptops, 12.1ms
Speed: 1.4ms preprocess, 

# 4. Analisando os Resultados da Detecção

## 4.1 Entendendo o Objeto Results
Quando chamamos `model(frame)`, recebemos uma lista de objetos `Results` (um para cada imagem/frame). O primeiro item, `results[0]`, contém todos os dados da detecção atual.

Os principais atributos que utilizaremos são:
- `results[0].boxes.xyxy`: As coordenadas da caixa (x inicial, y inicial, x final, y final).
- `results[0].boxes.cls`: O ID numérico da classe detectada.
- `results[0].boxes.conf`: A probabilidade (0 a 1) de ser o objeto correto.
- `model.names`: Um dicionário que mapeia o ID numérico para o nome legível (ex: 0 -> 'pessoa').

In [4]:
# Exemplo de como iterar sobre as caixas detectadas (use isso dentro do loop ou após uma inferência)
results = model(frame)
boxes = results[0].boxes

if boxes is not None:
    for box in boxes:
        # Obter ID da classe e converter para inteiro
        cls_id = int(box.cls[0])
        
        # Obter confiança da predição
        conf = float(box.conf[0])
        
        # Obter coordenadas da caixa
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        
        # Obter o nome da classe usando o dicionário do modelo
        class_name = model.names[cls_id]

        print(f"Classe: {class_name}, Confiança: {conf:.2f}, BBox: ({x1}, {y1}) -> ({x2}, {y2})")


0: 480x640 5 persons, 1 chair, 1 dining table, 1 laptop, 12.7ms
Speed: 2.4ms preprocess, 12.7ms inference, 1.6ms postprocess per image at shape (1, 3, 480, 640)
Classe: person, Confiança: 0.90, BBox: (152, 112) -> (637, 479)
Classe: person, Confiança: 0.87, BBox: (0, 242) -> (65, 370)
Classe: laptop, Confiança: 0.65, BBox: (234, 299) -> (311, 333)
Classe: person, Confiança: 0.61, BBox: (248, 263) -> (283, 300)
Classe: person, Confiança: 0.57, BBox: (37, 275) -> (62, 313)
Classe: person, Confiança: 0.40, BBox: (0, 243) -> (65, 475)
Classe: dining table, Confiança: 0.38, BBox: (0, 366) -> (182, 479)
Classe: chair, Confiança: 0.28, BBox: (72, 353) -> (221, 479)


## 4.2 Tabela de Referência das Classes (Dataset COCO)
Abaixo está a lista completa de classes que o modelo consegue identificar, junto com seus respectivos IDs. Use esta tabela para filtrar objetos específicos no seu código.

| ID | Classe | ID | Classe | ID | Classe | ID | Classe |
|---|---|---|---|---|---|---|---|
| 0 | person | 20 | elephant | 40 | wine glass | 60 | dining table |
| 1 | bicycle | 21 | bear | 41 | cup | 61 | toilet |
| 2 | car | 22 | zebra | 42 | fork | 62 | tv |
| 3 | motorcycle | 23 | giraffe | 43 | knife | 63 | laptop |
| 4 | airplane | 24 | backpack | 44 | spoon | 64 | mouse |
| 5 | bus | 25 | umbrella | 45 | bowl | 65 | remote |
| 6 | train | 26 | handbag | 46 | banana | 66 | keyboard |
| 7 | truck | 27 | tie | 47 | apple | 67 | cell phone |
| 8 | boat | 28 | suitcase | 48 | sandwich | 68 | microwave |
| 9 | traffic light | 29 | frisbee | 49 | orange | 69 | oven |
| 10 | fire hydrant | 30 | skis | 50 | broccoli | 70 | toaster |
| 11 | stop sign | 31 | snowboard | 51 | carrot | 71 | sink |
| 12 | parking meter | 32 | sports ball | 52 | hot dog | 72 | refrigerator |
| 13 | bench | 33 | kite | 53 | pizza | 73 | book |
| 14 | bird | 34 | baseball bat | 54 | donut | 74 | clock |
| 15 | cat | 35 | baseball glove | 55 | cake | 75 | vase |
| 16 | dog | 36 | skateboard | 56 | chair | 76 | scissors |
| 17 | horse | 37 | surfboard | 57 | couch | 77 | teddy bear |
| 18 | sheep | 38 | tennis racket | 58 | potted plant | 78 | hair drier |
| 19 | cow | 39 | bottle | 59 | bed | 79 | toothbrush |

# 5. Aplicação Prática: Sistema de Alerta

Agora que sabemos extrair os dados e conhecemos os IDs das classes, podemos criar regras lógicas. Nesta seção, vamos construir um sistema que monitora a webcam e dispara uma ação específica quando detecta um 'celular' (ID 67) com alta confiança.

## 5.1 Lógica do Alerta
A lógica será: SE a classe for 'cell phone' E a confiança for maior que 0.7 (70%), ENTÃO imprimir um alerta no console e salvar uma foto do momento.

In [5]:
import time
import cv2
from ultralytics import YOLO

# Carregar modelo
model = YOLO('yolo11n.pt')

# Iniciar webcam
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Erro ao acessar a webcam.")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Inferência
    results = model(frame)
    boxes = results[0].boxes

    # Verificar detecções
    if boxes is not None:
        for box in boxes:
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            class_name = model.names[cls_id]

            # Regra de Alerta: Celular (ID 67) com confiança > 70%
            if class_name == "cell phone" and conf > 0.7:
                print(f"ALERTA: Celular detectado! Confiança: {conf:.2f}")
                
                # Gerar nome de arquivo único com timestamp
                timestamp = time.strftime("%Y%m%d-%H%M%S")
                filename = f"alerta_celular_{timestamp}.jpg"
                
                # Salvar a imagem
                cv2.imwrite(filename, frame)
                print(f"Imagem salva como: {filename}")

    # Exibir vídeo com anotações padrão do YOLO
    annotated_frame = results[0].plot()
    cv2.imshow("Detecção com Alerta", annotated_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


0: 480x640 5 persons, 1 dining table, 2 laptops, 8.4ms
Speed: 1.1ms preprocess, 8.4ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 5 persons, 1 chair, 1 dining table, 1 laptop, 7.0ms
Speed: 1.0ms preprocess, 7.0ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 5 persons, 1 chair, 1 dining table, 1 laptop, 6.9ms
Speed: 1.2ms preprocess, 6.9ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 5 persons, 1 chair, 1 dining table, 1 laptop, 7.2ms
Speed: 1.0ms preprocess, 7.2ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 5 persons, 1 chair, 1 dining table, 3 laptops, 7.1ms
Speed: 1.0ms preprocess, 7.1ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 5 persons, 1 chair, 1 dining table, 3 laptops, 7.0ms
Speed: 1.0ms preprocess, 7.0ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 5 persons, 1 chair, 1 dining table

# 6. Conclusão

## 6.1 Resumo
Neste notebook, exploramos o uso do YOLOv8 para detecção de objetos. Vimos como o modelo nos fornece dados ricos sobre o que está vendo, permitindo não apenas visualizar, mas também programar ações baseadas no mundo real.

## 6.2 Próximos Passos
- Experimente detectar outras classes da tabela acima (ex: 'person'=0, 'cup'=41, 'laptop'=63).
- Tente controlar algo externo (como um LED via Arduino) quando um objeto for detectado.
- Treine seu próprio modelo para detectar objetos personalizados que não existem no dataset padrão.